In [1]:
import pandas as pd
import numpy as np

PARQUET_URL = "https://huggingface.co/datasets/sjelassi/new_omi_code_100k/resolve/main/data/train-00000-of-00001.parquet"

# Load the full parquet dataset
df = pd.read_parquet(PARQUET_URL)

print("=" * 60)
print(f"DATASET OVERVIEW")
print("=" * 60)
print(f"Total Rows:     {len(df):,}")
print(f"Total Columns:  {len(df.columns)}")
print(f"Memory Usage:   {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")

print("\n" + "=" * 60)
print("COLUMN DETAILS & DATA TYPES")
print("=" * 60)
info_df = pd.DataFrame({
    "Dtype": df.dtypes,
    "Non-Null Count": df.notnull().sum(),
    "Null Count": df.isnull().sum(),
    "Null %": (df.isnull().sum() / len(df) * 100).round(2),
    "Unique Values": df.nunique()
})
print(info_df)

print("\n" + "=" * 60)
print("TARGET VARIABLE DISTRIBUTION ('quality_score')")
print("=" * 60)
if "quality_score" in df.columns:
    q = df["quality_score"]
    print(q.describe())
    print(f"\nZero Variance Check: Standard Deviation = {q.std():.4f}")
    print(f"Distinct Quality Values: {q.nunique()}")
    print("\nTop 5 Most Frequent Quality Scores:")
    print(q.value_counts(normalize=True).head(5) * 100)
else:
    print("WARNING: 'quality_score' column not found in dataset!")

print("\n" + "=" * 60)
print("SAMPLE RECORD PREVIEW")
print("=" * 60)
sample_row = df.sample(1, random_state=42).iloc[0]
for col in df.columns:
    val = str(sample_row[col])
    # Truncate long code or prompt strings for clean displaying
    display_val = val[:120] + "..." if len(val) > 120 else val
    print(f"{col:20s}: {display_val}")

DATASET OVERVIEW
Total Rows:     100,000
Total Columns:  13
Memory Usage:   270.31 MB

COLUMN DETAILS & DATA TYPES
                        Dtype  Non-Null Count  Null Count  Null %  \
question               object          100000           0     0.0   
answer                 object          100000           0     0.0   
unit_tests             object          100000           0     0.0   
domain                 object          100000           0     0.0   
generation_algorithm   object          100000           0     0.0   
avg_test_score        float64          100000           0     0.0   
pass_rate             float64          100000           0     0.0   
test_count              int64          100000           0     0.0   
quality_score         float64          100000           0     0.0   
total_tokens            int64          100000           0     0.0   
def_count               int64          100000           0     0.0   
has_docstring            bool          100000           0

In [28]:
import pandas as pd
import numpy as np
import re
import ast
import joblib
from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from lightgbm import LGBMRegressor

RANDOM_STATE = 48

# ---------------------------------------------------------
# 1. Dataset Loading & Target Transformation
# ---------------------------------------------------------
PARQUET_URL = "https://huggingface.co/datasets/sjelassi/new_omi_code_100k/resolve/main/data/train-00000-of-00001.parquet"
print("Loading dataset...")
df = pd.read_parquet(PARQUET_URL)

subs = df.sample(n=20000, random_state=RANDOM_STATE).reset_index(drop=True)
subs = subs.dropna(subset=["quality_score"]).copy()

# Map continuous raw score to [0.0, 1.0]
subs["target_raw"] = (subs["quality_score"] - 15.1) / (15.3 - 15.1)
subs["target_raw"] = subs["target_raw"].clip(0.0, 1.0)

# Quantile Rank Transform
rank_pcts = subs["target_raw"].rank(method="average", pct=True).values

# ---------------------------------------------------------
# 2. Advanced AST & Structural Feature Engineering
# ---------------------------------------------------------
def clean_code(text):
    text = str(text).strip()
    text = re.sub(r"^```[a-zA-Z]*\n?", "", text)
    return re.sub(r"\n?```$", "", text).strip()

subs["clean_answer"] = subs["answer"].apply(clean_code)

def extract_advanced_ast_features(code_str):
    try:
        tree = ast.parse(code_str)
        nodes = list(ast.walk(tree))
        num_nodes = len(nodes)

        func_defs = sum(1 for n in nodes if isinstance(n, ast.FunctionDef))
        class_defs = sum(1 for n in nodes if isinstance(n, ast.ClassDef))
        loops = sum(1 for n in nodes if isinstance(n, (ast.For, ast.While)))
        conditionals = sum(1 for n in nodes if isinstance(n, ast.If))
        tries = sum(1 for n in nodes if isinstance(n, ast.Try))
        returns = sum(1 for n in nodes if isinstance(n, ast.Return))
        assigns = sum(1 for n in nodes if isinstance(n, (ast.Assign, ast.AugAssign)))
        type_annotations = sum(1 for n in nodes if isinstance(n, ast.AnnAssign))
        comprehensions = sum(1 for n in nodes if isinstance(n, (ast.ListComp, ast.DictComp, ast.SetComp, ast.GeneratorExp)))
        args_count = sum(len(n.args.args) for n in nodes if isinstance(n, ast.FunctionDef))

        # Track single-letter identifiers vs snake_case names
        identifiers = [n.id for n in nodes if isinstance(n, ast.Name)]
        single_char_vars = sum(1 for i in identifiers if len(i) == 1)
        total_vars = max(len(identifiers), 1)
        single_char_ratio = single_char_vars / total_vars

        # Estimate Cyclomatic Complexity (Control flow decisions + 1)
        cyclomatic_complexity = 1 + conditionals + loops + tries + sum(
            1 for n in nodes if isinstance(n, ast.BoolOp)
        )

        # AST Tree Max Depth
        def get_depth(node):
            children = list(ast.iter_child_nodes(node))
            if not children:
                return 1
            return 1 + max(get_depth(child) for child in children)

        ast_depth = get_depth(tree)

        # Check for nested loops
        nested_loops = 0
        for n in nodes:
            if isinstance(n, (ast.For, ast.While)):
                for child in ast.walk(n):
                    if child is not n and isinstance(child, (ast.For, ast.While)):
                        nested_loops += 1
                        break

        syntax_valid = 1
    except Exception:
        num_nodes, func_defs, class_defs, loops, conditionals, tries, returns = 0, 0, 0, 0, 0, 0, 0
        assigns, type_annotations, comprehensions, args_count, cyclomatic_complexity = 0, 0, 0, 0, 0
        single_char_ratio, ast_depth, nested_loops = 0.0, 0, 0
        syntax_valid = 0

    return [
        num_nodes, func_defs, class_defs, loops, conditionals, tries,
        returns, assigns, type_annotations, comprehensions, args_count,
        cyclomatic_complexity, single_char_ratio, ast_depth, nested_loops, syntax_valid
    ]

print("Extracting advanced AST features...")
ast_feats = np.array([extract_advanced_ast_features(c) for c in subs["clean_answer"]], dtype=np.float32)

# Extract Code Surface Ratios & Advanced Statistics
print("Calculating code surface ratios and complexity metrics...")
subs["code_len"] = subs["clean_answer"].str.len().astype(np.float32)
subs["line_count"] = (subs["clean_answer"].str.count(r"\n") + 1).astype(np.float32)
subs["has_docstring"] = subs["has_docstring"].astype(np.float32)
subs["pass_rate"] = subs["pass_rate"].fillna(0.5).astype(np.float32) if "pass_rate" in subs else 0.5
subs["test_count"] = subs["test_count"].fillna(0.0).astype(np.float32) if "test_count" in subs else 0.0

# Surface ratios
subs["avg_line_length"] = (subs["code_len"] / np.maximum(subs["line_count"], 1.0)).astype(np.float32)
subs["comment_density"] = (subs["clean_answer"].str.count(r"#") / np.maximum(subs["line_count"], 1.0)).astype(np.float32)
subs["indentation_spaces"] = (subs["clean_answer"].str.count(r"    ") / np.maximum(subs["line_count"], 1.0)).astype(np.float32)
subs["operator_count"] = subs["clean_answer"].str.count(r"[\+\-\*\/\%\|\&\^\=\!\<\>]").astype(np.float32)
subs["operator_density"] = (subs["operator_count"] / np.maximum(subs["code_len"], 1.0)).astype(np.float32)

# AST Ratios
node_counts = np.maximum(ast_feats[:, 0], 1.0)
line_counts = np.maximum(subs["line_count"].values, 1.0)
ast_density = (ast_feats[:, 0] / line_counts).astype(np.float32)
ast_complexity_ratio = (ast_feats[:, 11] / node_counts).astype(np.float32)

engineered_ratios = np.column_stack([ast_density, ast_complexity_ratio])

basic_num_cols = [
    "has_docstring", "code_len", "line_count", "pass_rate", "test_count",
    "avg_line_length", "comment_density", "indentation_spaces", "operator_count", "operator_density"
]
X_basic_num = subs[basic_num_cols].values.astype(np.float32)

# Combine numerical feature arrays
X_all_num = np.hstack([X_basic_num, ast_feats, engineered_ratios])

num_feature_names = basic_num_cols + [
    "ast_num_nodes", "ast_func_defs", "ast_class_defs", "ast_loops",
    "ast_conditionals", "ast_tries", "ast_returns", "ast_assigns",
    "ast_type_annotations", "ast_comprehensions", "ast_args_count",
    "ast_cyclomatic_complexity", "ast_single_char_ratio", "ast_depth",
    "ast_nested_loops", "ast_syntax_valid", "ast_density_per_line",
    "ast_complexity_per_node"
]

# ---------------------------------------------------------
# 3. Dual TF-IDF Feature Extraction (Word + Char N-Grams)
# ---------------------------------------------------------
print("Extracting dual (Word + Character) TF-IDF features...")

# Character n-grams (3-5)
tfidf_char = TfidfVectorizer(max_features=600, analyzer="char_wb", ngram_range=(3, 5))
X_text_char = tfidf_char.fit_transform(subs["clean_answer"]).astype(np.float32)
char_feature_names = [f"tfidf_char_{i}" for i in range(X_text_char.shape[1])]

# Word n-grams (1-2) with raw string token_pattern
tfidf_word = TfidfVectorizer(max_features=400, analyzer="word", ngram_range=(1, 2), token_pattern=r"(?u)\b\w+\b")
X_text_word = tfidf_word.fit_transform(subs["clean_answer"]).astype(np.float32)
word_feature_names = [f"tfidf_word_{i}" for i in range(X_text_word.shape[1])]

all_feature_names = num_feature_names + char_feature_names + word_feature_names

# Construct combined sparse feature matrix
X_sparse = hstack([csr_matrix(X_all_num), X_text_char, X_text_word]).tocsr().astype(np.float32)
y = rank_pcts.astype(np.float32)

# ---------------------------------------------------------
# 4. Train / Validation / Test Splits
# ---------------------------------------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X_sparse, y, test_size=0.2, random_state=48
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.3, random_state=RANDOM_STATE
)

# ---------------------------------------------------------
# 5. LightGBM Regressor Training
# ---------------------------------------------------------
print("Training feature-enhanced LightGBM Regressor...")
model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.02,
    num_leaves=90,
    max_depth=12,
    subsample=0.85,
    colsample_bytree=0.75,
    min_child_samples=20,
    random_state=RANDOM_STATE,
    verbosity=-1
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="rmse"
)

# ---------------------------------------------------------
# 6. Evaluation & Artifact Export
# ---------------------------------------------------------
X_test_df = pd.DataFrame.sparse.from_spmatrix(X_test, columns=all_feature_names)

preds = model.predict(X_test_df)
preds_clipped = np.clip(preds, 0.0, 1.0)

print("\n" + "=" * 50)
print("FEATURE-ENHANCED LIGHTGBM EVALUATION")
print("=" * 50)
print(f"R2 Score: {r2_score(y_test, preds_clipped):.4f}")
print(f"RMSE:     {np.sqrt(mean_squared_error(y_test, preds_clipped)):.4f}")
print(f"MAE:      {mean_absolute_error(y_test, preds_clipped):.4f}")
print(f"Min Score Predicted: {preds_clipped.min():.4f}")
print(f"Max Score Predicted: {preds_clipped.max():.4f}")

ARTIFACT_NAME = "code_grading_lgbm.joblib"
joblib.dump(
    {
        "model": model,
        "tfidf_char": tfidf_char,
        "tfidf_word": tfidf_word,
        "feature_names": all_feature_names
    },
    ARTIFACT_NAME
)
print(f"\nArtifact exported successfully to {ARTIFACT_NAME}")

Loading dataset...
Extracting advanced AST features...


<unknown>:11: SyntaxWarning: invalid escape sequence '\s'


Calculating code surface ratios and complexity metrics...
Extracting dual (Word + Character) TF-IDF features...
Training feature-enhanced LightGBM Regressor...

FEATURE-ENHANCED LIGHTGBM EVALUATION
R2 Score: 0.4309
RMSE:     0.1791
MAE:      0.1038
Min Score Predicted: 0.0159
Max Score Predicted: 0.9629

Artifact exported successfully to code_grading_lgbm.joblib
